# 13 — PROJECT DATA EXPLORATION & VISUAL DISCOVERY

# 00. CONFIGURATION & SOURCE INVENTORY

In [1]:
# ============================================================
# NB_12 — PROJECT DATA EXPLORATION & VISUAL DISCOVERY
# 00. CONFIGURATION & SOURCE INVENTORY
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")

PATHS = {
    "original": PROJECT_ROOT / "Data_Original",
    "source_outputs": PROJECT_ROOT / "03_Outputs",
    "planning_outputs": PROJECT_ROOT / "05_Outputs",
    "final_outputs": PROJECT_ROOT / "07_Outputs",
}

# Directorio exclusivo para este laboratorio
EXPLORATION_DIR = PROJECT_ROOT / "Exploration_Visual_Discovery"
EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. EXPECTED SOURCE FILES
# ------------------------------------------------------------

EXPECTED_FILES = {
    "Original Project": (
        PATHS["original"] / "C2011-13 Wind Farm.xlsx"
    ),
    "Activity Tracking": (
        PATHS["source_outputs"] / "Activity_Tracking.xlsx"
    ),
    "Activity Master with Materials": (
        PATHS["planning_outputs"] / "Activity_Master_with_Materials.xlsx"
    ),
    "Schedule Control": (
        PATHS["final_outputs"] / "Schedule_Control.xlsx"
    ),
    "Scenario Simulation": (
        PATHS["final_outputs"] / "Scenario_Simulation_Results.xlsx"
    ),
    "Corrective Actions": (
        PATHS["final_outputs"] / "Corrective_Action_Evaluation.xlsx"
    ),
    "Economic Impact": (
        PATHS["final_outputs"] / "Economic_Impact_Assessment.xlsx"
    ),
}

# ------------------------------------------------------------
# 3. INVENTORY
# ------------------------------------------------------------

inventory = []

for name, filepath in EXPECTED_FILES.items():

    exists = filepath.exists()

    inventory.append({
        "Source": name,
        "File": filepath.name,
        "Exists": exists,
        "Size_KB": (
            round(filepath.stat().st_size / 1024, 2)
            if exists else np.nan
        ),
        "Path": str(filepath)
    })

df_source_inventory = pd.DataFrame(inventory)

print("=" * 65)
print("NB_12 — DATA EXPLORATION & VISUAL DISCOVERY")
print("=" * 65)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Exploration directory: {EXPLORATION_DIR}")

display(df_source_inventory)

print("\nTOTAL SOURCES:", len(df_source_inventory))
print("AVAILABLE:", df_source_inventory["Exists"].sum())
print("MISSING:", (~df_source_inventory["Exists"]).sum())

**Estructura del cuaderno**

| Bloque | Contenido |
| :--- | :--- |
| **00** | Configuración, rutas y carga de fuentes |
| **01** | Inventario, estructura y calidad de datos |
| **02** | Evolución semanal del proyecto |
| **03** | Earned Value: PV, EV, AC, SPI, CPI y SV(t) |
| **04** | Actividades: avance, duración y desviaciones |
| **05** | Gantt, ruta crítica y holguras |
| **06** | Suministros, dependencias y exposición |
| **07** | Escenarios y acciones correctivas |
| **08** | Relaciones cruzadas y hallazgos |
| **09** | Catálogo de visualizaciones y conclusiones |

## **Primera investigación: histórico semanal**

Empezaría por aquí porque puede proporcionarnos la mayor riqueza analítica.

Queremos averiguar:
- Cómo evoluciona el avance de las 167 actividades.
- Si el ritmo de ejecución es estable o presenta discontinuidades.
- Cómo evolucionan PV, EV y AC.
- Cuándo aparecen desviaciones y si persisten.
- Qué actividades explican las desviaciones globales.
- Si existen diferencias importantes entre grupos de actividades.
- Si los indicadores agregados ocultan problemas individuales.
Y una precaución: comprobaremos la procedencia y significado del histórico antes de interpretarlo como ejecución real.

In [2]:
# ============================================================
# NB_12 — PROJECT DATA EXPLORATION & VISUAL DISCOVERY
# 00. CONFIGURATION & SOURCE INVENTORY
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")

PATHS = {
    "original": PROJECT_ROOT / "Data_Original",
    "source_outputs": PROJECT_ROOT / "03_Outputs",
    "planning_outputs": PROJECT_ROOT / "05_Outputs",
    "final_outputs": PROJECT_ROOT / "07_Outputs",
}

# Directorio exclusivo para este laboratorio
EXPLORATION_DIR = PROJECT_ROOT / "Exploration_Visual_Discovery"
EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. EXPECTED SOURCE FILES
# ------------------------------------------------------------

EXPECTED_FILES = {
    "Original Project": (
        PATHS["original"] / "C2011-13 Wind Farm.xlsx"
    ),
    "Activity Tracking": (
        PATHS["source_outputs"] / "Activity_Tracking.xlsx"
    ),
    "Activity Master with Materials": (
        PATHS["planning_outputs"] / "Activity_Master_with_Materials.xlsx"
    ),
    "Schedule Control": (
        PATHS["final_outputs"] / "Schedule_Control.xlsx"
    ),
    "Scenario Simulation": (
        PATHS["final_outputs"] / "Scenario_Simulation_Results.xlsx"
    ),
    "Corrective Actions": (
        PATHS["final_outputs"] / "Corrective_Action_Evaluation.xlsx"
    ),
    "Economic Impact": (
        PATHS["final_outputs"] / "Economic_Impact_Assessment.xlsx"
    ),
}

# ------------------------------------------------------------
# 3. INVENTORY
# ------------------------------------------------------------

inventory = []

for name, filepath in EXPECTED_FILES.items():

    exists = filepath.exists()

    inventory.append({
        "Source": name,
        "File": filepath.name,
        "Exists": exists,
        "Size_KB": (
            round(filepath.stat().st_size / 1024, 2)
            if exists else np.nan
        ),
        "Path": str(filepath)
    })

df_source_inventory = pd.DataFrame(inventory)

print("=" * 65)
print("NB_12 — DATA EXPLORATION & VISUAL DISCOVERY")
print("=" * 65)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Exploration directory: {EXPLORATION_DIR}")

display(df_source_inventory)

print("\nTOTAL SOURCES:", len(df_source_inventory))
print("AVAILABLE:", df_source_inventory["Exists"].sum())
print("MISSING:", (~df_source_inventory["Exists"]).sum())

# 01. WORKBOOK & DATA STRUCTURE AUDIT

In [3]:
# ============================================================
# NB_12 — 01. WORKBOOK & DATA STRUCTURE AUDIT
# ============================================================

from openpyxl import load_workbook

workbook_inventory = []
column_inventory = {}

for source_name, filepath in EXPECTED_FILES.items():

    if not filepath.exists():
        continue

    print("\n" + "=" * 90)
    print(f"SOURCE: {source_name}")
    print(f"FILE: {filepath.name}")
    print("=" * 90)

    try:
        xls = pd.ExcelFile(filepath, engine="openpyxl")

        for sheet_name in xls.sheet_names:

            df = pd.read_excel(
                filepath,
                sheet_name=sheet_name,
                engine="openpyxl"
            )

            workbook_inventory.append({
                "Source": source_name,
                "Sheet": sheet_name,
                "Rows": len(df),
                "Columns": len(df.columns),
                "Empty_Rows": int(df.isna().all(axis=1).sum()),
                "Empty_Columns": int(df.isna().all(axis=0).sum())
            })

            column_inventory[(source_name, sheet_name)] = {
                "columns": df.columns.tolist(),
                "dtypes": df.dtypes.astype(str).to_dict()
            }

            print(f"\n[{sheet_name}]")
            print(f"Dimensions: {df.shape}")
            print("Columns:")

            for col in df.columns:
                print(f"  - {col} [{df[col].dtype}]")

    except Exception as e:
        print(f"ERROR: {e}")

# ------------------------------------------------------------
# CONSOLIDATED INVENTORY
# ------------------------------------------------------------

df_workbook_inventory = pd.DataFrame(workbook_inventory)

print("\n\n" + "=" * 90)
print("CONSOLIDATED WORKBOOK INVENTORY")
print("=" * 90)

display(
    df_workbook_inventory.sort_values(
        ["Source", "Sheet"]
    ).reset_index(drop=True)
)

print("\nTotal sheets:", len(df_workbook_inventory))
print("Total rows across sheets:", df_workbook_inventory["Rows"].sum())

# 02. WEEKLY TRACKING & HISTORICAL COVERAGE AUDIT

In [4]:
# ============================================================
# NB_12 — 02. WEEKLY TRACKING & HISTORICAL COVERAGE AUDIT
# ============================================================

import re

tracking_path = EXPECTED_FILES["Activity Tracking"]
original_path = EXPECTED_FILES["Original Project"]
master_path = EXPECTED_FILES["Activity Master with Materials"]

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df_tracking = pd.read_excel(tracking_path, sheet_name="Sheet1")
df_master_explore = pd.read_excel(master_path, sheet_name="Sheet1")

# ------------------------------------------------------------
# 2. BASIC STRUCTURE
# ------------------------------------------------------------

print("=" * 75)
print("TRACKING DATA — BASIC STRUCTURE")
print("=" * 75)

print("Rows:", len(df_tracking))
print("Activities:", df_tracking["Activity_ID"].nunique())
print("Tracking periods:", df_tracking["TP"].nunique())
print("Date range:",
      df_tracking["Status_Date"].min(),
      "to",
      df_tracking["Status_Date"].max())

print("\nUnique TP values:")
print(sorted(df_tracking["TP"].dropna().unique()))

# ------------------------------------------------------------
# 3. ORIGINAL WORKBOOK — TP SHEET INVENTORY
# ------------------------------------------------------------

xls_original = pd.ExcelFile(original_path, engine="openpyxl")

tp_sheets = [
    sheet for sheet in xls_original.sheet_names
    if re.fullmatch(r"TP\s*\d+", sheet.strip(), flags=re.IGNORECASE)
]

tp_sheet_inventory = []

for sheet in tp_sheets:

    df_sheet = pd.read_excel(
        original_path,
        sheet_name=sheet,
        engine="openpyxl"
    )

    tp_sheet_inventory.append({
        "Sheet": sheet,
        "Rows": len(df_sheet),
        "Columns": len(df_sheet.columns),
        "Empty": df_sheet.empty,
        "Column_Names": " | ".join(map(str, df_sheet.columns))
    })

df_tp_sheets = pd.DataFrame(tp_sheet_inventory)

print("\n" + "=" * 75)
print("ORIGINAL WORKBOOK — TP SHEETS")
print("=" * 75)

print("TP sheets found:", len(df_tp_sheets))
display(df_tp_sheets.head(10))
display(df_tp_sheets.tail(5))

# ------------------------------------------------------------
# 4. RECORDS PER TRACKING PERIOD
# ------------------------------------------------------------

df_tp_summary = (
    df_tracking.groupby("TP")
    .agg(
        Records=("Activity_ID", "size"),
        Unique_Activities=("Activity_ID", "nunique"),
        Date_Min=("Status_Date", "min"),
        Date_Max=("Status_Date", "max"),
        Unique_Dates=("Status_Date", "nunique"),
        Progress_Mean=("Pct_Complete", "mean"),
        Progress_Min=("Pct_Complete", "min"),
        Progress_Max=("Pct_Complete", "max"),
        EV_Total=("EV", "sum"),
        PV_Total=("PV", "sum"),
        AC_Total=("Actual_Cost", "sum")
    )
    .reset_index()
    .sort_values("TP")
)

print("\n" + "=" * 75)
print("TRACKING PERIOD SUMMARY")
print("=" * 75)

display(df_tp_summary.head(10))
display(df_tp_summary.tail(10))

# ------------------------------------------------------------
# 5. KEY INTEGRITY AND COVERAGE
# ------------------------------------------------------------

expected_activity_ids = set(df_master_explore["ID"])
tracking_activity_ids = set(df_tracking["Activity_ID"])

duplicate_keys = df_tracking.duplicated(
    subset=["TP", "Activity_ID"],
    keep=False
)

coverage_by_tp = (
    df_tracking.groupby("TP")["Activity_ID"]
    .nunique()
)

quality_summary = pd.DataFrame({
    "Check": [
        "Expected activities in master",
        "Activities found in tracking",
        "Activities missing from tracking",
        "Tracking IDs not in master",
        "Expected records (TP × activities)",
        "Actual tracking records",
        "Duplicate TP-Activity records",
        "Missing Status_Date",
        "Missing Pct_Complete",
        "Missing EV",
        "Missing PV",
        "Missing Actual_Cost",
        "Missing Actual_Start",
        "Missing Actual_Duration",
        "Missing Actual_Duration_Days"
    ],
    "Value": [
        len(expected_activity_ids),
        len(tracking_activity_ids),
        len(expected_activity_ids - tracking_activity_ids),
        len(tracking_activity_ids - expected_activity_ids),
        df_tracking["TP"].nunique() * len(expected_activity_ids),
        len(df_tracking),
        int(duplicate_keys.sum()),
        int(df_tracking["Status_Date"].isna().sum()),
        int(df_tracking["Pct_Complete"].isna().sum()),
        int(df_tracking["EV"].isna().sum()),
        int(df_tracking["PV"].isna().sum()),
        int(df_tracking["Actual_Cost"].isna().sum()),
        int(df_tracking["Actual_Start"].isna().sum()),
        int(df_tracking["Actual_Duration"].isna().sum()),
        int(df_tracking["Actual_Duration_Days"].isna().sum())
    ]
})

print("\n" + "=" * 75)
print("DATA QUALITY & COVERAGE")
print("=" * 75)

display(quality_summary)

# ------------------------------------------------------------
# 6. VARIABLE DISTRIBUTIONS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("NUMERIC VARIABLE SUMMARY")
print("=" * 75)

display(
    df_tracking[
        [
            "Pct_Complete",
            "EV",
            "PV",
            "Actual_Cost",
            "Actual_Duration_Days"
        ]
    ].describe(percentiles=[.05, .25, .50, .75, .95]).T
)

print("\nActual_Duration — sample values:")
display(
    df_tracking["Actual_Duration"]
    .value_counts(dropna=False)
    .head(15)
    .to_frame("Records")
)

# ------------------------------------------------------------
# 7. DATE CONSISTENCY BY TP
# ------------------------------------------------------------

date_consistency = (
    df_tracking.groupby("TP")
    .agg(
        Min_Date=("Status_Date", "min"),
        Max_Date=("Status_Date", "max"),
        Distinct_Dates=("Status_Date", "nunique")
    )
    .reset_index()
)

date_consistency["Single_Date_Per_TP"] = (
    date_consistency["Distinct_Dates"] == 1
)

print("\n" + "=" * 75)
print("DATE CONSISTENCY BY TP")
print("=" * 75)

display(date_consistency)

print("\nAUDIT COMPLETED")

# 02B. RESUMEN COMPACTO DEL HISTÓRICO

In [5]:
# ============================================================
# NB_12 — 02B. RESUMEN COMPACTO DEL HISTÓRICO
# ============================================================

print("1. COBERTURA POR PERIODO")
display(
    df_tp_summary[
        ["TP", "Records", "Unique_Activities", "Date_Min",
         "Date_Max", "Progress_Mean", "EV_Total", "PV_Total", "AC_Total"]
    ].iloc[[0, 1, 2, 3, 4, -5, -4, -3, -2, -1]]
)

print("\n2. CALIDAD E INTEGRIDAD")
display(quality_summary)

print("\n3. DISTRIBUCIÓN DE VARIABLES")
display(
    df_tracking[
        ["Pct_Complete", "EV", "PV", "Actual_Cost", "Actual_Duration_Days"]
    ].describe(percentiles=[.05, .25, .50, .75, .95]).T
)

print("\n4. VALORES DE ACTUAL_DURATION")
display(
    df_tracking["Actual_Duration"]
    .value_counts(dropna=False)
    .head(12)
    .to_frame("Records")
)

print("\n5. CONSISTENCIA DE FECHAS")
display(
    date_consistency[
        ["TP", "Min_Date", "Max_Date", "Distinct_Dates", "Single_Date_Per_TP"]
    ].iloc[[0, 1, 2, 3, 4, -5, -4, -3, -2, -1]]
)

# 02C. AUDITORÍA DE IMPORTES POR NIVEL WBS

In [6]:
# ============================================================
# NB_12 — 02C. AUDITORÍA DE IMPORTES POR NIVEL WBS
# ============================================================

# 1. IDENTIFICAR ACTIVIDADES TERMINALES
df_schedule_explore = pd.read_excel(
    EXPECTED_FILES["Schedule Control"],
    sheet_name="Schedule Control"
)

terminal_ids = set(df_schedule_explore["ID"].dropna().astype(int))
all_ids = set(df_master_explore["ID"].dropna().astype(int))
summary_ids = all_ids - terminal_ids

df_tracking_audit = df_tracking.copy()
df_tracking_audit["Activity_Type"] = np.where(
    df_tracking_audit["Activity_ID"].isin(terminal_ids),
    "Terminal",
    "WBS Summary"
)

print("ACTIVITIES")
print("Total IDs:", len(all_ids))
print("Terminal activities:", len(terminal_ids))
print("WBS summary nodes:", len(summary_ids))

# 2. PRESUPUESTO DEL MAESTRO: TODOS LOS NODOS VS TERMINALES
budget_audit = pd.DataFrame({
    "Scope": ["All master records", "Terminal activities", "WBS summary nodes"],
    "Records": [
        len(df_master_explore),
        len(terminal_ids),
        len(summary_ids)
    ],
    "Planned_Cost_EUR": [
        df_master_explore["Total_Cost_Plan"].sum(),
        df_master_explore.loc[
            df_master_explore["ID"].isin(terminal_ids), "Total_Cost_Plan"
        ].sum(),
        df_master_explore.loc[
            df_master_explore["ID"].isin(summary_ids), "Total_Cost_Plan"
        ].sum()
    ]
})
display(budget_audit)

# 3. EV, PV Y AC: SEPARAR LOS DOS NIVELES
df_level_totals = (
    df_tracking_audit
    .groupby(["TP", "Activity_Type"])
    .agg(
        EV=("EV", "sum"),
        PV=("PV", "sum"),
        AC=("Actual_Cost", "sum"),
        Progress_Mean=("Pct_Complete", "mean"),
        Records=("Activity_ID", "size")
    )
    .reset_index()
)

df_level_pivot = df_level_totals.pivot(
    index="TP",
    columns="Activity_Type",
    values=["EV", "PV", "AC", "Progress_Mean"]
)

# 4. COMPARACIÓN EN PERIODOS REPRESENTATIVOS
selected_tps = [1, 2, 12, 26, 52, 78, 104, 116, 117, 118, 119, 120]

display(
    df_level_pivot.loc[selected_tps].round(2)
)

# 5. COMPROBAR SI LOS IMPORTES DE NODOS RESUMEN SON DUPLICADOS
df_duplication_check = (
    df_level_pivot[[("EV", "Terminal"), ("EV", "WBS Summary"),
                    ("PV", "Terminal"), ("PV", "WBS Summary"),
                    ("AC", "Terminal"), ("AC", "WBS Summary")]]
    .copy()
)

for metric in ["EV", "PV", "AC"]:
    df_duplication_check[(metric, "Summary_to_Terminal_Ratio")] = (
        df_duplication_check[(metric, "WBS Summary")] /
        df_duplication_check[(metric, "Terminal")].replace(0, np.nan)
    )

print("RATIOS WBS SUMMARY / TERMINAL")
display(
    df_duplication_check.loc[selected_tps].round(3)
)

# 02D. VALIDACIÓN DE LA SEMÁNTICA EVM

In [7]:
# ============================================================
# NB_12 — 02D. VALIDACIÓN DE LA SEMÁNTICA EVM
# ============================================================

# 1. HISTÓRICO SOLO DE ACTIVIDADES TERMINALES
df_terminal = df_tracking[
    df_tracking["Activity_ID"].isin(terminal_ids)
].copy()

df_terminal = df_terminal.merge(
    df_master_explore[
        ["ID", "Name", "Total_Cost_Plan", "Duration_Plan_Days", "WBS"]
    ],
    left_on="Activity_ID",
    right_on="ID",
    how="left",
    validate="many_to_one"
)

# 2. COMPROBAR SI EV = PRESUPUESTO × AVANCE
df_terminal["EV_Expected"] = (
    df_terminal["Total_Cost_Plan"] * df_terminal["Pct_Complete"]
)
df_terminal["EV_Difference"] = (
    df_terminal["EV"] - df_terminal["EV_Expected"]
)

print("A. EV VS PRESUPUESTO × AVANCE")
display(
    df_terminal[["EV", "EV_Expected", "EV_Difference"]]
    .describe(percentiles=[.5, .90, .95, .99]).T
)

print(
    "Correlación EV y presupuesto × avance:",
    df_terminal[["EV", "EV_Expected"]].corr().iloc[0, 1]
)

# 3. EV, PV Y AC AGREGADOS POR PERIODO
df_terminal_weekly = (
    df_terminal.groupby(["TP", "Status_Date"])
    .agg(
        EV=("EV", "sum"),
        PV=("PV", "sum"),
        AC=("Actual_Cost", "sum"),
        Progress_Mean=("Pct_Complete", "mean")
    )
    .reset_index()
    .sort_values("TP")
)

for col in ["EV", "PV", "AC"]:
    df_terminal_weekly[f"{col}_Change"] = df_terminal_weekly[col].diff()

print("\nB. EVOLUCIÓN AGREGADA — PERIODOS SELECCIONADOS")
display(
    df_terminal_weekly[
        df_terminal_weekly["TP"].isin(
            [1, 2, 12, 26, 52, 78, 104, 116, 117, 118, 119, 120]
        )
    ].round(2)
)

print("\nC. MONOTONICIDAD DE LOS IMPORTES")
for col in ["EV", "PV", "AC"]:
    changes = df_terminal_weekly[f"{col}_Change"].dropna()
    print(
        col,
        "| Incrementos negativos:", int((changes < -0.01).sum()),
        "| Incrementos cero:", int((changes.abs() <= 0.01).sum()),
        "| Incrementos positivos:", int((changes > 0.01).sum())
    )

# 4. RECONCILIACIÓN FINAL POR ACTIVIDAD
df_final = (
    df_terminal[df_terminal["TP"] == df_terminal["TP"].max()]
    .copy()
)

df_final["Cost_Variance"] = (
    df_final["Actual_Cost"] - df_final["Total_Cost_Plan"]
)
df_final["Cost_Variance_Pct"] = (
    df_final["Cost_Variance"] /
    df_final["Total_Cost_Plan"].replace(0, np.nan) * 100
)

print("\nD. RECONCILIACIÓN ECONÓMICA TP120")
print("Budget:", round(df_final["Total_Cost_Plan"].sum(), 2))
print("EV:", round(df_final["EV"].sum(), 2))
print("AC:", round(df_final["Actual_Cost"].sum(), 2))
print("Cost variance:", round(df_final["Cost_Variance"].sum(), 2))

print("\nE. ACTIVIDADES CON MAYOR DESVIACIÓN DE COSTE")
display(
    df_final[
        ["Activity_ID", "Name", "Total_Cost_Plan", "Pct_Complete",
         "EV", "Actual_Cost", "Cost_Variance", "Cost_Variance_Pct"]
    ]
    .sort_values("Cost_Variance", ascending=False)
    .head(15)
    .round(2)
)

# 03. EVOLUCIÓN SEMANAL DEL PROYECTO

In [8]:
# ============================================================
# NB_12 — 03. EVOLUCIÓN SEMANAL DEL PROYECTO
# Exploración visual: avance, planificación y coste
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Agregación semanal de actividades terminales
df_dashboard_weekly = (
    df_tracking[
        df_tracking["Activity_ID"].isin(terminal_ids)
    ]
    .groupby(["TP", "Status_Date"], as_index=False)
    .agg(
        EV=("EV", "sum"),
        PV=("PV", "sum"),
        AC=("Actual_Cost", "sum")
    )
    .sort_values("TP")
)

# Indicadores de seguimiento
df_dashboard_weekly["SPI"] = (
    df_dashboard_weekly["EV"] /
    df_dashboard_weekly["PV"].replace(0, np.nan)
)

df_dashboard_weekly["CPI"] = (
    df_dashboard_weekly["EV"] /
    df_dashboard_weekly["AC"].replace(0, np.nan)
)

# Conversión a millones para facilitar la lectura
for col in ["EV", "PV", "AC"]:
    df_dashboard_weekly[f"{col}_M"] = (
        df_dashboard_weekly[col] / 1_000_000
    )

# ============================================================
# VISUALIZACIÓN
# ============================================================

fig, axes = plt.subplots(
    3, 1,
    figsize=(15, 13),
    sharex=True
)

# 1. Evolución económica acumulada
axes[0].plot(
    df_dashboard_weekly["Status_Date"],
    df_dashboard_weekly["PV_M"],
    label="PV — Planned Value",
    linewidth=2
)

axes[0].plot(
    df_dashboard_weekly["Status_Date"],
    df_dashboard_weekly["EV_M"],
    label="EV — Earned Value",
    linewidth=2
)

axes[0].plot(
    df_dashboard_weekly["Status_Date"],
    df_dashboard_weekly["AC_M"],
    label="AC — Actual Cost",
    linewidth=2
)

axes[0].set_title("Evolución acumulada: planificación, avance y coste")
axes[0].set_ylabel("Millones de €")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Desviación de plazo en valor
axes[1].plot(
    df_dashboard_weekly["Status_Date"],
    (df_dashboard_weekly["EV"] - df_dashboard_weekly["PV"]) / 1_000_000,
    linewidth=2
)

axes[1].axhline(0, linestyle="--", linewidth=1)
axes[1].set_title("Schedule Variance (EV − PV)")
axes[1].set_ylabel("Millones de €")
axes[1].grid(alpha=0.3)

# 3. Índices de eficiencia
axes[2].plot(
    df_dashboard_weekly["Status_Date"],
    df_dashboard_weekly["SPI"],
    label="SPI",
    linewidth=2
)

axes[2].plot(
    df_dashboard_weekly["Status_Date"],
    df_dashboard_weekly["CPI"],
    label="CPI",
    linewidth=2
)

axes[2].axhline(1, linestyle="--", linewidth=1)
axes[2].set_title("Índices de rendimiento: SPI y CPI")
axes[2].set_ylabel("Índice")
axes[2].set_xlabel("Fecha de seguimiento")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 03B. LECTURA NUMÉRICA DE LA EVOLUCIÓN SEMANAL

In [9]:
# ============================================================
# NB_12 — 03B. LECTURA NUMÉRICA DE LA EVOLUCIÓN SEMANAL
# ============================================================

# 1. VALORES EN PERIODOS REPRESENTATIVOS
selected_periods = [1, 12, 26, 52, 78, 104, 116, 117, 118, 119, 120]

print("1. EVOLUCIÓN EN PERIODOS SELECCIONADOS")
display(
    df_dashboard_weekly[
        df_dashboard_weekly["TP"].isin(selected_periods)
    ][
        ["TP", "Status_Date", "PV_M", "EV_M", "AC_M",
         "SPI", "CPI"]
    ].round(3)
)

# 2. DESVIACIÓN DE PLAZO Y COSTE
df_dashboard_weekly["SV_M"] = (
    df_dashboard_weekly["EV"] - df_dashboard_weekly["PV"]
) / 1_000_000

df_dashboard_weekly["CV_M"] = (
    df_dashboard_weekly["EV"] - df_dashboard_weekly["AC"]
) / 1_000_000

print("\n2. RESUMEN DE DESVIACIONES")
display(
    df_dashboard_weekly[
        ["SV_M", "CV_M", "SPI", "CPI"]
    ].describe(percentiles=[.10, .25, .50, .75, .90]).round(3)
)

# 3. PERIODOS DE MAYOR Y MENOR DESVIACIÓN
for col, description in [
    ("SV_M", "Desviación de plazo (SV)"),
    ("CV_M", "Desviación de coste (CV)"),
    ("SPI", "Índice de plazo (SPI)"),
    ("CPI", "Índice de coste (CPI)")
]:
    valid = df_dashboard_weekly.dropna(subset=[col])

    print(f"\n{description}")
    print("Mínimo:")
    display(valid.loc[[valid[col].idxmin()],
                      ["TP", "Status_Date", col]])

    print("Máximo:")
    display(valid.loc[[valid[col].idxmax()],
                      ["TP", "Status_Date", col]])

# 4. CAMBIOS SEMANALES: DETECCIÓN DE ACELERACIONES Y DETERIOROS
for col in ["EV_M", "PV_M", "AC_M", "SV_M", "CV_M"]:
    df_dashboard_weekly[f"{col}_Change"] = (
        df_dashboard_weekly[col].diff()
    )

print("\n4. MAYORES CAMBIOS SEMANALES")
for col in ["EV_M_Change", "PV_M_Change", "AC_M_Change",
            "SV_M_Change", "CV_M_Change"]:
    print(f"\n{col}")
    display(
        df_dashboard_weekly.nlargest(5, col)[
            ["TP", "Status_Date", col]
        ].round(3)
    )

    print("Mayores caídas:")
    display(
        df_dashboard_weekly.nsmallest(5, col)[
            ["TP", "Status_Date", col]
        ].round(3)
    )

# 5. PRIMERA Y ÚLTIMA SEMANA CON INDICADORES DISPONIBLES
print("\n5. VALORES INICIALES Y FINALES")
display(
    df_dashboard_weekly.iloc[
        [0, -1]
    ][
        ["TP", "Status_Date", "PV_M", "EV_M", "AC_M",
         "SV_M", "CV_M", "SPI", "CPI"]
    ].round(3)
)

# 03C. PERIODOS DE DESVIACIÓN Y RECUPERACIÓN

In [10]:
# ============================================================
# NB_12 — 03C. PERIODOS DE DESVIACIÓN Y RECUPERACIÓN
# ============================================================

df_periods = df_dashboard_weekly.sort_values("TP").copy()

# Clasificación semanal
df_periods["Schedule_Below_1"] = df_periods["SPI"] < 1
df_periods["Cost_Below_1"] = df_periods["CPI"] < 1

def get_periods(df, condition_col):
    """Agrupa semanas consecutivas que cumplen una condición."""
    data = df[["TP", "Status_Date", condition_col]].copy()

    data["Episode"] = (
        data[condition_col] != data[condition_col].shift()
    ).cumsum()

    result = (
        data[data[condition_col]]
        .groupby("Episode")
        .agg(
            TP_Start=("TP", "min"),
            TP_End=("TP", "max"),
            Date_Start=("Status_Date", "min"),
            Date_End=("Status_Date", "max"),
            Weeks=("TP", "count")
        )
        .reset_index(drop=True)
    )

    return result

# 1. EPISODIOS DE RETRASO
schedule_periods = get_periods(
    df_periods, "Schedule_Below_1"
)

print("1. EPISODIOS CON SPI < 1")
display(schedule_periods)

# 2. EPISODIOS DE INEFICIENCIA DE COSTE
cost_periods = get_periods(
    df_periods, "Cost_Below_1"
)

print("\n2. EPISODIOS CON CPI < 1")
display(cost_periods)

# 3. RESUMEN DE DURACIÓN Y RECUPERACIÓN
print("\n3. RESUMEN")

for name, periods in [
    ("SPI", schedule_periods),
    ("CPI", cost_periods)
]:
    print(f"\n{name}")
    print("Semanas desfavorables:",
          int(periods["Weeks"].sum()))
    print("Número de episodios:",
          len(periods))

    if not periods.empty:
        longest = periods.loc[periods["Weeks"].idxmax()]
        print("Episodio más largo:",
              int(longest["Weeks"]), "semanas")
        print("Desde TP", int(longest["TP_Start"]),
              "hasta TP", int(longest["TP_End"]))

# 4. HITOS DE RECUPERACIÓN DEL SPI Y CPI
print("\n4. PRIMERA SEMANA CON ÍNDICE >= 1 TRAS SU MÍNIMO")

for metric in ["SPI", "CPI"]:
    minimum_idx = df_periods[metric].idxmin()
    minimum = df_periods.loc[minimum_idx]

    recovered = df_periods[
        (df_periods["TP"] > minimum["TP"]) &
        (df_periods[metric] >= 1)
    ]

    print(f"\n{metric}")
    print("Mínimo:", round(minimum[metric], 3),
          "| TP:", int(minimum["TP"]),
          "| Fecha:", minimum["Status_Date"])

    if not recovered.empty:
        first = recovered.iloc[0]
        print("Primera recuperación:",
              round(first[metric], 3),
              "| TP:", int(first["TP"]),
              "| Fecha:", first["Status_Date"])
    else:
        print("No recupera el índice durante el periodo observado.")

# 5. EVOLUCIÓN DEL TRAMO FINAL: TP78-TP120
print("\n5. TRAMO FINAL")
display(
    df_periods[
        df_periods["TP"].between(78, 120)
    ][
        ["TP", "Status_Date", "EV_M", "PV_M", "AC_M",
         "SV_M", "CV_M", "SPI", "CPI"]
    ].iloc[::4].round(3)
)

In [19]:
# ============================================================
# NB_12 — 04. ACTIVIDADES QUE EXPLICAN LAS DESVIACIONES
# ============================================================

# Histórico de actividades terminales
df_activity_analysis = df_tracking[
    df_tracking["Activity_ID"].isin(terminal_ids)
].merge(
    df_master_explore[
        ["ID", "Name", "WBS", "Total_Cost_Plan"]
    ],
    left_on="Activity_ID",
    right_on="ID",
    how="left",
    validate="many_to_one"
)

# Desviaciones por actividad y periodo
df_activity_analysis["Schedule_Gap"] = (
    df_activity_analysis["PV"] - df_activity_analysis["EV"]
)

df_activity_analysis["Cost_Gap"] = (
    df_activity_analysis["Actual_Cost"] - df_activity_analysis["EV"]
)

# Periodos de interés
periods = {
    66: "Peor SPI",
    80: "Peor CPI",
    100: "Mayor retraso acumulado",
    120: "Cierre del proyecto"
}

for tp, description in periods.items():

    df_tp = df_activity_analysis[
        df_activity_analysis["TP"] == tp
    ].copy()

    print("\n" + "=" * 75)
    print(f"TP{tp} — {description}")
    print("=" * 75)

    # Principales actividades con retraso respecto al plan
    print("\nA. MAYORES CONTRIBUYENTES AL RETRASO (PV - EV)")

    display(
        df_tp[df_tp["Schedule_Gap"] > 0]
        .nlargest(10, "Schedule_Gap")[
            ["Activity_ID", "Name", "WBS", "Pct_Complete",
             "PV", "EV", "Schedule_Gap"]
        ].round(2)
    )

    # Principales desviaciones económicas
    print("\nB. MAYORES DESVIACIONES DE COSTE (AC - EV)")

    display(
        df_tp[df_tp["Cost_Gap"] > 0]
        .nlargest(10, "Cost_Gap")[
            ["Activity_ID", "Name", "WBS", "Total_Cost_Plan",
             "Pct_Complete", "EV", "Actual_Cost", "Cost_Gap"]
        ].round(2)
    )

    # Concentración de las desviaciones
    positive_schedule = df_tp["Schedule_Gap"].clip(lower=0).sum()
    positive_cost = df_tp["Cost_Gap"].clip(lower=0).sum()

    print("\nC. MAGNITUD AGREGADA")
    print(f"Retraso bruto por actividades: {positive_schedule:,.2f} €")
    print(f"Desviación positiva bruta de coste: {positive_cost:,.2f} €")
    print(f"Desviación neta de coste (AC - EV): "
          f"{df_tp['Cost_Gap'].sum():,.2f} €")

In [20]:
# ============================================================
# 04B. CONCENTRACIÓN Y EVOLUCIÓN DE LOS SOBRECOSTES
# ============================================================

# 1. PARETO DE DESVIACIONES EN TP120
df_final_cost = (
    df_activity_analysis[
        df_activity_analysis["TP"] == 120
    ]
    .groupby(["Activity_ID", "Name", "WBS"], as_index=False)
    .agg(
        Budget=("Total_Cost_Plan", "first"),
        EV=("EV", "sum"),
        AC=("Actual_Cost", "sum")
    )
)

df_final_cost["Cost_Gap"] = df_final_cost["AC"] - df_final_cost["EV"]

df_pareto = (
    df_final_cost[df_final_cost["Cost_Gap"] > 0]
    .sort_values("Cost_Gap", ascending=False)
    .copy()
)

df_pareto["Cumulative_Gap"] = df_pareto["Cost_Gap"].cumsum()
df_pareto["Cumulative_Share"] = (
    df_pareto["Cumulative_Gap"] /
    df_pareto["Cost_Gap"].sum()
)

print("A. PARETO DE SOBRECOSTES — TP120")
display(
    df_pareto[
        ["Activity_ID", "Name", "WBS", "Budget",
         "EV", "AC", "Cost_Gap", "Cumulative_Share"]
    ].head(20).round(2)
)

print("\nConcentración del sobrecoste positivo:")
for n in [1, 3, 5, 10, 15, 20]:
    share = df_pareto.head(n)["Cost_Gap"].sum() / df_pareto["Cost_Gap"].sum()
    print(f"Top {n:2d}: {share:.1%}")

# 2. EVOLUCIÓN DE LOS PRINCIPALES FOCOS
focus_groups = {
    "Roads and hardstands": [20],
    "WEC 1-7": list(range(94, 101)),
    "Building of the cabin": [153],
    "Installation towers": [53, 59, 65, 71, 77, 83, 89]
}

rows = []

for group, activity_ids in focus_groups.items():

    temp = df_activity_analysis[
        df_activity_analysis["Activity_ID"].isin(activity_ids)
    ]

    weekly = (
        temp.groupby(["TP", "Status_Date"], as_index=False)
        .agg(
            EV=("EV", "sum"),
            PV=("PV", "sum"),
            AC=("Actual_Cost", "sum")
        )
    )

    weekly["Group"] = group
    weekly["Cost_Gap_M"] = (weekly["AC"] - weekly["EV"]) / 1_000_000
    weekly["Schedule_Gap_M"] = (weekly["PV"] - weekly["EV"]) / 1_000_000

    rows.append(weekly)

df_focus_evolution = pd.concat(rows, ignore_index=True)

# 3. COMPARACIÓN EN PERIODOS CLAVE
selected = [26, 52, 66, 80, 100, 120]

print("\nB. EVOLUCIÓN DE LOS FOCOS — MILLONES DE €")
display(
    df_focus_evolution[
        df_focus_evolution["TP"].isin(selected)
    ][
        ["Group", "TP", "Cost_Gap_M", "Schedule_Gap_M"]
    ]
    .pivot(index="Group", columns="TP",
           values=["Cost_Gap_M", "Schedule_Gap_M"])
    .round(3)
)

# 4. GRÁFICO DE EVOLUCIÓN DEL SOBRECOSTE
fig, ax = plt.subplots(figsize=(14, 6))

for group in focus_groups:
    temp = df_focus_evolution[
        df_focus_evolution["Group"] == group
    ]

    ax.plot(
        temp["TP"],
        temp["Cost_Gap_M"],
        label=group,
        linewidth=2
    )

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Evolución del sobrecoste por principales focos")
ax.set_xlabel("Periodo de seguimiento (TP)")
ax.set_ylabel("AC − EV (millones de €)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Hallazgos que conservaría para el dashboard
1. El sobrecoste está muy concentrado.
- La actividad 20, Roads and hardstands, explica el 45 % del sobrecoste positivo.
- Las diez actividades con mayor desviación explican el 90,7 %.
- Roads and hardstands y WEC 1–7, conjuntamente, representan aproximadamente el 86,8 % del sobrecoste positivo.
2. Hay dos comportamientos económicos diferentes.
- Roads and hardstands: acumula 2,179 M€ de desviación ya en TP80 y la mantiene hasta el cierre. Es un sobrecoste que aparece pronto y persiste.
- WEC 1–7: su desviación pasa de 0,579 M€ en TP100 a 2,026 M€ en TP120. El sobrecoste se concentra en la fase final.
3. La recuperación del calendario tiene protagonistas.

WEC 1–7 acumula 6,112 M€ de retraso respecto al plan en TP100 y llega a cero en TP120. Las instalaciones de torre también reducen sustancialmente su desviación.

Esto nos da una historia muy buena para el dashboard: qué paquetes provocaron el retraso, cuáles permitieron recuperar el calendario y dónde quedó el sobrecoste.

In [21]:
# ============================================================
# NB_12 — 04C. TRAYECTORIA DE EJECUCIÓN POR PAQUETE
# ============================================================

focus_activities = {
    "Roads and hardstands": [20],
    "WEC 1-7": list(range(94, 101)),
    "Installation towers": [53, 59, 65, 71, 77, 83, 89],
    "Building of the cabin": [153]
}

rows = []

for group, ids in focus_activities.items():

    temp = df_activity_analysis[
        df_activity_analysis["Activity_ID"].isin(ids)
    ].copy()

    weekly = (
        temp.groupby("TP")
        .agg(
            Activities_Started=("Pct_Complete", lambda x: (x > 0).sum()),
            Activities_Completed=("Pct_Complete", lambda x: (x >= 1).sum()),
            Progress_Mean=("Pct_Complete", "mean"),
            EV=("EV", "sum"),
            PV=("PV", "sum"),
            AC=("Actual_Cost", "sum")
        )
        .reset_index()
    )

    weekly["Group"] = group
    weekly["Schedule_Gap_M"] = (weekly["PV"] - weekly["EV"]) / 1_000_000
    weekly["Cost_Gap_M"] = (weekly["AC"] - weekly["EV"]) / 1_000_000

    rows.append(weekly)

df_execution_groups = pd.concat(rows, ignore_index=True)

# Resumen de hitos de ejecución
summary = []

for group, temp in df_execution_groups.groupby("Group"):

    temp = temp.sort_values("TP")

    started = temp[temp["Activities_Started"] > 0]
    completed = temp[
        temp["Activities_Completed"] == temp["Activities_Completed"].max()
    ]

    summary.append({
        "Group": group,
        "Activities": int(temp["Activities_Started"].max()),
        "First_Activity_Started_TP": (
            int(started["TP"].min()) if not started.empty else np.nan
        ),
        "First_Activity_Completed_TP": (
            int(temp.loc[temp["Activities_Completed"] > 0, "TP"].min())
            if (temp["Activities_Completed"] > 0).any() else np.nan
        ),
        "All_Completed_TP": (
            int(temp.loc[
                temp["Activities_Completed"] == temp["Activities_Started"].max(),
                "TP"
            ].min())
            if (temp["Activities_Completed"] == temp["Activities_Started"].max()).any()
            else np.nan
        ),
        "Progress_TP100": (
            float(temp.loc[temp["TP"] == 100, "Progress_Mean"].iloc[0])
            if (temp["TP"] == 100).any() else np.nan
        ),
        "Progress_TP120": float(temp.iloc[-1]["Progress_Mean"])
    })

df_execution_summary = pd.DataFrame(summary)

print("A. HITOS DE EJECUCIÓN POR PAQUETE")
display(df_execution_summary.round(3))

# Evolución del avance medio
fig, ax = plt.subplots(figsize=(14, 6))

for group in focus_activities:
    temp = df_execution_groups[
        df_execution_groups["Group"] == group
    ]

    ax.plot(
        temp["TP"],
        temp["Progress_Mean"] * 100,
        label=group,
        linewidth=2
    )

ax.set_title("Evolución del avance de los principales paquetes")
ax.set_xlabel("Periodo de seguimiento (TP)")
ax.set_ylabel("Avance medio (%)")
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Tres conclusiones relevantes:
1. Roads and hardstands termina en TP80, antes de que comiencen las instalaciones de torre en TP88.
2. En TP100, las torres ya alcanzan un avance medio del 84,3 %, mientras que WEC 1–7 solo alcanza el 28,6 %. Los aerogeneradores son el paquete que mantiene actividad más avanzada la ejecución.
3. Los cuatro paquetes están completados en TP120. Sin embargo, terminar el trabajo no significa haberlo hecho dentro del presupuesto: Roads y WEC concentran la mayor parte del sobrecoste final.
Hay una distinción importante: estos datos describen la secuencia y el avance registrado, pero no demuestran por sí solos que un paquete haya causado el retraso de otro.

In [22]:
# ============================================================
# NB_12 — 05. EXPOSICIÓN Y RIESGO DE SUMINISTROS
# ============================================================

df_supply = pd.read_excel(
    PATHS["final_outputs"] / "Schedule_Control.xlsx"
)

# Actividades con exposición de suministro
supply_cols = [
    "ID", "Name", "WBS",
    "Material_Type",
    "Lead_Time_Days",
    "Buffer_Days",
    "Planned_Anticipation_Days",
    "Supply_Delay_Days",
    "Material_Status",
    "Supply_Risk_Score",
    "Material_Alarm",
    "Supply_Buffer_Excess_Days",
    "Supply_Buffer_Exceeded",
    "Supply_Alert",
    "Schedule_Risk",
    "Risk_Level",
    "Schedule_Exposure"
]

df_supply_exposure = df_supply[
    [c for c in supply_cols if c in df_supply.columns]
].copy()

# Excluir actividades sin información de suministro
df_supply_exposure = df_supply_exposure[
    df_supply_exposure["Material_Type"].notna()
].copy()

print("A. ACTIVIDADES CON INFORMACIÓN DE SUMINISTRO")
print(f"Actividades identificadas: {len(df_supply_exposure)}")

display(
    df_supply_exposure.sort_values(
        ["Supply_Risk_Score", "Supply_Buffer_Excess_Days"],
        ascending=False
    ).round(2)
)

print("\nB. RESUMEN DE EXPOSICIÓN")

summary_supply = {
    "Activities_with_supply_data": len(df_supply_exposure),
    "Activities_with_supply_delay": int(
        (df_supply_exposure["Supply_Delay_Days"].fillna(0) > 0).sum()
    ),
    "Activities_exceeding_buffer": int(
        df_supply_exposure["Supply_Buffer_Exceeded"].fillna(False).sum()
    ),
    "Activities_with_supply_alert": int(
        df_supply_exposure["Supply_Alert"].fillna(False).sum()
    )
}

display(pd.Series(summary_supply, name="Value").to_frame())

- 135 actividades tienen información de suministro.
- 38 (28,1 %) presentan retraso y generan una alerta.
- 9 (6,7 % del total; 23,7 % de las alertas) superan su buffer.
- Las otras 29 alertas todavía no han superado el buffer.
La distinción es importante: no debemos presentar las 38 alertas como 38 retrasos del proyecto. Una alerta identifica una exposición que requiere seguimiento; superar el buffer indica que el retraso de suministro ya rebasa la protección prevista.

Además, el campo Schedule_Exposure indica que puede existir holgura disponible. Por tanto, necesitamos cruzar estas alertas con la holgura CPM antes de calificarlas como amenaza directa a la fecha final.

In [24]:
# ============================================================
# NB_12 — 05B. SUMINISTROS VS. HOLGURA CPM
# ============================================================

# Incorporar holgura y criticidad CPM
df_supply_risk = df_supply_exposure.merge(
    df_supply[["ID", "Total_Float", "Is_Critical"]],
    on="ID",
    how="left",
    validate="one_to_one"
)

# Trabajar únicamente con actividades que presentan retraso
df_alerts = df_supply_risk[
    df_supply_risk["Supply_Delay_Days"].fillna(0) > 0
].copy()

df_alerts["Total_Float"] = pd.to_numeric(
    df_alerts["Total_Float"], errors="coerce"
)

# Clasificación según holgura disponible
df_alerts["Float_Category"] = np.select(
    [
        df_alerts["Total_Float"].isna(),
        df_alerts["Total_Float"] <= 0,
        df_alerts["Total_Float"] < df_alerts["Supply_Delay_Days"],
    ],
    [
        "Unknown float",
        "Critical / zero float",
        "Delay exceeds float",
    ],
    default="Float absorbs delay"
)

print("A. ALERTAS POR SITUACIÓN DE HOLGURA")

display(
    df_alerts.groupby(
        ["Float_Category", "Supply_Buffer_Exceeded"],
        dropna=False
    ).agg(
        Activities=("ID", "count"),
        Max_Supply_Delay=("Supply_Delay_Days", "max"),
        Max_Buffer_Excess=("Supply_Buffer_Excess_Days", "max")
    ).reset_index()
)

print("\nB. ALERTAS CON BUFFER SUPERADO")

display(
    df_alerts[df_alerts["Supply_Buffer_Exceeded"] == True][
        [
            "ID", "Name", "Material_Type",
            "Supply_Delay_Days", "Buffer_Days",
            "Supply_Buffer_Excess_Days",
            "Total_Float", "Is_Critical",
            "Schedule_Exposure", "Risk_Level"
        ]
    ].sort_values(
        "Supply_Buffer_Excess_Days", ascending=False
    ).round(1)
)

print("\nC. RESUMEN DE ALERTAS")

print(f"Alertas totales: {len(df_alerts)}")
print(
    "Con buffer superado:",
    int(df_alerts["Supply_Buffer_Exceeded"].fillna(False).sum())
)
print(
    "Con holgura cero o negativa:",
    int((df_alerts["Total_Float"] <= 0).sum())
)
print(
    "Con retraso superior a la holgura:",
    int(
        (
            df_alerts["Supply_Delay_Days"]
            > df_alerts["Total_Float"]
        ).fillna(False).sum()
    )
)

Los datos permiten distinguir tres niveles:
Situación	Actividades	Interpretación
Alerta sin superar buffer	29	Retraso dentro del buffer previsto
Buffer superado, holgura disponible	9	Incumplimiento de la protección de suministro, pero absorbible según la holgura CPM
Retraso superior a holgura CPM	0	No se identifica impacto directo potencial sobre las fechas calculadas

El caso más significativo es la actividad 79, injection of tower:
- Retraso de suministro: 35 días.
- Buffer previsto: 21 días.
- Exceso sobre buffer: 14 días.
- Holgura CPM: 179 días.
No es una actividad crítica por calendario, pero sí presenta una desviación relevante respecto a la planificación de suministro.

Una cautela importante: la holgura CPM es una referencia del calendario base. No demuestra que cualquier retraso pueda absorberse sin consecuencias operativas; habría que considerar también dependencias reales, disponibilidad de recursos y posibles restricciones de ejecución.
Qué significa para el dashboard
No presentaría un semáforo único de riesgo de suministro. Utilizaría dos indicadores separados:
- Riesgo de suministro: retrasos y buffers superados.
- Exposición al calendario: holgura disponible y actividades críticas afectadas.
Esto evita que nueve alertas de suministro parezcan nueve amenazas a la fecha final del proyecto.

In [26]:
# ============================================================
# NB_12 — 05C. CRUCE DE ALERTAS DE SUMINISTRO Y EJECUCIÓN
# ============================================================

# Actividades con retraso de suministro
alert_ids = df_alerts["ID"].unique()

# Histórico de esas actividades
df_supply_history = df_activity_analysis[
    df_activity_analysis["Activity_ID"].isin(alert_ids)
].copy()

# Resumen por actividad en el último periodo
last_period = (
    df_supply_history[df_supply_history["TP"] == 120]
    .groupby(["Activity_ID", "Name"], as_index=False)
    .agg(
        Progress=("Pct_Complete", "mean"),
        EV=("EV", "sum"),
        PV=("PV", "sum"),
        AC=("Actual_Cost", "sum")
    )
)

last_period["Schedule_Gap"] = last_period["PV"] - last_period["EV"]
last_period["Cost_Gap"] = last_period["AC"] - last_period["EV"]

# Incorporar información de suministro y CPM
df_supply_execution = last_period.merge(
    df_alerts[
        [
            "ID", "Supply_Delay_Days",
            "Buffer_Days", "Supply_Buffer_Excess_Days",
            "Total_Float", "Risk_Level"
        ]
    ],
    left_on="Activity_ID",
    right_on="ID",
    how="left",
    validate="one_to_one"
)

print("A. ACTIVIDADES CON ALERTA DE SUMINISTRO — TP120")

display(
    df_supply_execution[
        [
            "Activity_ID", "Name", "Progress",
            "Schedule_Gap", "Cost_Gap",
            "Supply_Delay_Days",
            "Supply_Buffer_Excess_Days",
            "Total_Float", "Risk_Level"
        ]
    ].sort_values(
        ["Supply_Buffer_Excess_Days", "Supply_Delay_Days"],
        ascending=False
    ).round(2)
)

print("\nB. RESUMEN")

print(
    "Actividades con avance inferior al 100 %:",
    int((df_supply_execution["Progress"] < 1).sum())
)
print(
    "Actividades con desviación de calendario positiva:",
    int((df_supply_execution["Schedule_Gap"] > 0).sum())
)
print(
    "Actividades con desviación de coste positiva:",
    int((df_supply_execution["Cost_Gap"] > 0).sum())
)

as alertas de suministro no se traducen en retrasos pendientes de ejecución al cierre del histórico.

En TP120:
- Las 38 actividades con alerta están completadas al 100 %.
- Ninguna conserva desviación de calendario (PV − EV = 0).
- 33 de las 38 presentan desviación de coste positiva.
- Las nueve que superaron el buffer también están completadas y no presentan retraso pendiente.
Esto cambia la interpretación de la página de suministros: no debemos mostrar estas alertas como riesgos actuales de ejecución, sino como incidencias de suministro registradas durante el proyecto, con su exposición y resultado final.

Hay, además, una distinción que conviene conservar:
- El retraso de suministro fue real dentro del modelo de planificación.
- La holgura CPM permitió absorberlo en el calendario calculado.
- El histórico termina con las actividades completadas.
- No podemos atribuir los sobrecostes a esos retrasos solo porque coincidan en las mismas actividades.
Mi conclusión sobre la vista de suministros
No necesita una página independiente. La integraría en Schedule & Execution, con tres elementos:
1. Indicador de actividades con retraso de suministro: 38.
2. Indicador de buffers superados: 9.
3. Tabla priorizada con actividad, retraso, buffer excedido y holgura CPM.
Y añadiría una nota de contexto: todas las actividades afectadas constan completadas en TP120; no hay desviación de calendario pendiente en ellas.